# Four-card tensor-core correctness gate — CMP 170HX, 2026-09-02

| Metric | Value |
|---|---|
| Cards passed | 4 / 4 |
| gpu-burn -tc range | 74.6 - 78.5 TFLOP/s |
| Peak core / memory temp (worst card) | 63 C / 71 C |
| Matmul max abs err (BF16/FP16/TF32/INT8) vs. CPU float64 | 1.348 / 0.166 / 0.098 / 0, identical across all four cards |

![chart](../assets/charts/2026-09-02-cmp170hx-health-gate.png)

```bash
scripts/qc/tensor-gate.sh 0 1 2 3
```

[results/2026-09-02-health-gate/README.md](../results/2026-09-02-health-gate/README.md)

In [1]:
# --- Status cell -------------------------------------------------------
# LIVE = False replays the committed receipts under results/<experiment>/.
# LIVE = True re-runs the gate against live hardware (needs a GPU host,
# not this notebook's normal execution path).
import os

EXPERIMENT = "2026-09-02-health-gate"
RESULTS_DIR = os.path.join("..", "results", EXPERIMENT, "receipts")
LIVE = False

print(f"experiment  : {EXPERIMENT}")
print(f"results_dir : {RESULTS_DIR}")
print(f"LIVE        : {LIVE}")


experiment  : 2026-09-02-health-gate
results_dir : ../results/2026-09-02-health-gate/receipts
LIVE        : False


# TL;DR

Four cards, one gate: 10-minute `gpu-burn -tc`, a deterministic BF16/FP16/TF32/INT8
matmul cross-check, a full-VRAM `memtest_vulkan` pass, and a PCIe replay/AER
snapshot. This run followed an out-of-memory incident that took the whole
fleet down to a UVM fatal error and a VM reboot; the gate is the recovery
verification, not a routine check.

In [2]:
import json
import os

GPUS = [0, 1, 2, 3]

def load_json(rel):
    with open(os.path.join(RESULTS_DIR, rel)) as f:
        return json.load(f)

receipts = {g: load_json(f"gpu{g}/receipt.json") for g in GPUS}
matmuls = {g: load_json(f"gpu{g}/matmul.json") for g in GPUS}
baseline = load_json("day2-baseline.json")
burn_summary = {c["gpu"]: c for c in load_json("burn-tc-summary.json")["cards"]}

for g in GPUS:
    print(f"gpu{g}: stop_reason={receipts[g]['stop_reason']}, xid_count={receipts[g]['xid_count_during_card']}")


gpu0: stop_reason=none, xid_count=0
gpu1: stop_reason=none, xid_count=0
gpu2: stop_reason=none, xid_count=0
gpu3: stop_reason=none, xid_count=0


In [3]:
from IPython.display import display, Markdown

def render_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |",
             "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    display(Markdown("\n".join(lines)))

rows = []
for g in GPUS:
    r = receipts[g]
    m = matmuls[g]["paths"]
    b = burn_summary[g]
    all_clean = (
        r["burn_tc_rc"] == 0 and r["matmul_rc"] == 0 and r["memtest_rc"] == 0
        and r["xid_count_during_card"] == 0 and r["stop_reason"] == "none"
    )
    rows.append([
        f"GPU {g}",
        f"{b['gflops']/1000:.1f}",
        r["xid_count_during_card"],
        f"{r['final_core_temp_c']} / {r['final_memory_temp_c']}",
        "PASS" if r["memtest_rc"] == 0 else "FAIL",
        f"{m['bf16']['max_abs_err_vs_cpu_f64']:.3f} / {m['fp16']['max_abs_err_vs_cpu_f64']:.3f} / "
        f"{m['tf32']['max_abs_err_vs_cpu_f64']:.3f} / {m['int8']['max_abs_err_vs_cpu_i64']}",
        "PASS" if all_clean else "CHECK",
    ])

render_table(
    ["GPU", "gpu-burn (TFLOP/s)", "Xid during gate", "Core/mem temp peak (C)",
     "memtest_vulkan", "BF16/FP16/TF32/INT8 max abs err", "Verdict"],
    rows,
)


| GPU | gpu-burn (TFLOP/s) | Xid during gate | Core/mem temp peak (C) | memtest_vulkan | BF16/FP16/TF32/INT8 max abs err | Verdict |
|---|---|---|---|---|---|---|
| GPU 0 | 77.7 | 0 | 62 / 64 | PASS | 1.348 / 0.166 / 0.098 / 0 | PASS |
| GPU 1 | 76.4 | 0 | 56 / 64 | PASS | 1.348 / 0.166 / 0.098 / 0 | PASS |
| GPU 2 | 74.6 | 0 | 63 / 71 | PASS | 1.348 / 0.166 / 0.098 / 0 | PASS |
| GPU 3 | 78.5 | 0 | 58 / 65 | PASS | 1.348 / 0.166 / 0.098 / 0 | PASS |

Pins: driver 610.43.03, kernel 6.8.0-138-generic, `gpu-burn -tc` (10 min per
card), `memtest_vulkan` v0.5.0 full-VRAM pass, matmul check on PyTorch
2.13.0+cu130 (seed 170170, 4096x4096). Full identity table in
[results/2026-09-02-health-gate/README.md](../results/2026-09-02-health-gate/README.md#hardware).

## Visible results

The chart above pairs each card's `gpu-burn -tc` throughput with its peak
core and memory temperature during the gate. All four cards clear 74
TFLOP/s and stay well under the 80 C core / 85 C memory stop conditions.

In [4]:
baseline_rows = []
for card in baseline["cards"]:
    baseline_rows.append([
        card["index"],
        f"{card['temp_core_c']} / {card['temp_memory_c']}",
        card["power_draw_w"],
        card["pcie_link_status"],
        len(card["xid_since_boot"]),
    ])

render_table(
    ["GPU", "Idle core/mem temp (C)", "Idle power (W)", "PCIe link", "Xid entries since boot"],
    baseline_rows,
)


| GPU | Idle core/mem temp (C) | Idle power (W) | PCIe link | Xid entries since boot |
|---|---|---|---|---|
| 0 | 38 / 41 | 33.62 | Gen1 x8 (board topology, expected) | 0 |
| 1 | 37 / 50 | 37.9 | Gen1 x16 (ok) | 0 |
| 2 | 38 / 51 | 33.94 | Gen1 x16 (ok) | 0 |
| 3 | 38 / 51 | 41.4 | Gen1 x16 (ok) | 1 |

This baseline was collected read-only, right after the post-incident VM
reboot and before the gate ran. One card carries a software-class Xid 43
from a wedge earlier the same day (see the appendix); it still ran the
gate clean.

## Reproduce

**Hardware.** 4x CMP 170HX, 64 GiB each, forced airflow, 180 W power cap per
card. One card runs at PCIe x8 due to board lane sharing — see
[docs/HARDWARE.md](../docs/HARDWARE.md); this is expected and not a fault.

**Prerequisites.**

- `gpu-burn` and `memtest_vulkan` built or placed next to the script (see
  [scripts/qc/README.md](../scripts/qc/README.md#what-is-still-external) —
  neither binary is committed to this repository).
- A Python interpreter with `torch` and CUDA available, pointed to by
  `TG_MATMUL_PYTHON` if the system interpreter lacks `torch`.
- No resident serving container, no coordination lock file, and zero
  compute processes fleet-wide — the script refuses to start otherwise.

**Run.**

```bash
TG_MATMUL_PYTHON=<path-to-venv>/bin/python3 scripts/qc/tensor-gate.sh 0 1 2 3
```

Expect roughly 15-20 minutes per card (10-minute burn plus matmul, memtest,
and PCIe snapshot stages), run sequentially, so four cards take about an
hour to an hour and a half end to end.

**Verify.** Each card writes a JSON receipt to
`logs/tensor-gate-<timestamp>/receipts/gpu<N>/`. A clean run shows
`xid_count_during_card: 0` and matching matmul errors across all four
cards; see [docs/QC.md](../docs/QC.md#stage-3b-tensor-core-correctness-gate)
for the full stage description and refusal conditions.

## Appendix

<details>
<summary>Incident timeline, script fixes, and limitations</summary>

### Incident that triggered this gate

1. **Fleet-wide OOM (~14:50 UTC).** A weight-load out-of-memory event during
   a load test hit all four pipeline-parallel ranks at once.
2. **Xid 31 on two cards.** MMU faults on the two cards holding the affected
   pipeline-parallel workers.
3. **Xid 154 on all four cards.** A UVM global fatal error (recovery action
   "OS Reboot") followed on every card. A full `nvidia`/`nvidia_uvm` module
   reload did not recover the fleet.
4. **VM reboot,** required to restore the fleet.
5. **This gate,** run on all four cards before trusting the fleet with a new
   benchmark publication.

Full account: [docs/FAILURE-RISK.md](../docs/FAILURE-RISK.md#fleet-wide-oom-can-escalate-to-a-required-vm-reboot).

### Script bugs found and fixed in `tensor-gate.sh`

1. `gpu-burn`'s binary path and compare-kernel fatbin were not found from
   the script's working directory; fixed with an explicit `-c <path>`.
2. `memtest_vulkan` has no `--top-fraction` flag and no CLI device-select
   flag — device choice is an interactive prompt that ignores piped stdin,
   so a naive call silently tests the wrong card. Fixed with
   `memtest-select.py`, a PTY wrapper that matches the target card's PCI
   bus ID to the tool's own listed index.
3. The matmul stage's system Python lacked `torch`; reran with a venv
   interpreter that had `torch 2.13.0+cu130`.

A fourth issue, found during setup rather than during this run: orphaned
`memtest_vulkan` processes from earlier manual testing pinned VRAM on all
four cards, because the binary re-execs into a detached grandchild that a
plain `SIGKILL` does not reach. Fixed by signaling the whole process group
plus a `pkill -f` backstop.

### Limitations

This gate does not exercise sustained multi-hour operation, mixed-workload
memory pressure, or the exact conditions of the OOM cascade that preceded
it. A clean result here is evidence of no persisting damage under this
gate's own profile, not a blanket guarantee against a fault that only
reproduces under a different workload.

</details>